In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

df = pd.read_csv('../data/European_Bank.csv')
df = df.drop(columns=['Year'])

df['AgeBand'] = pd.cut(df['Age'], bins=[18, 30, 45, 60, 92], labels=['18-30', '31-45', '46-60', '61+'])
df['HighBalance'] = (df['Balance'] > df['Balance'].median()).astype(int)

def classify_engagement(row):
    if row['IsActiveMember'] == 1:
        if row['NumOfProducts'] >= 2:
            return 'Active Engaged'
        else:
            return 'Active Low-Product'
    else:
        if row['HighBalance'] == 1:
            return 'Inactive High-Balance'
        else:
            return 'Inactive Disengaged'

df['EngagementProfile'] = df.apply(classify_engagement, axis=1)
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,AgeBand,HighBalance,EngagementProfile
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,31-45,0,Active Low-Product
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,31-45,0,Active Low-Product
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,31-45,1,Inactive High-Balance
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,31-45,0,Inactive Disengaged
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,31-45,1,Active Low-Product


In [2]:
churn_active = df[df['IsActiveMember']==1]['Exited'].mean()
churn_inactive = df[df['IsActiveMember']==0]['Exited'].mean()

engagement_retention_ratio = churn_inactive / churn_active
print(f"Active churn: {churn_active:.4f}")
print(f"Inactive churn: {churn_inactive:.4f}")
print(f"Engagement Retention Ratio: {engagement_retention_ratio:.2f}")

Active churn: 0.1427
Inactive churn: 0.2685
Engagement Retention Ratio: 1.88


In [3]:
product_stats = df.groupby('NumOfProducts').agg(
    Count=('Exited', 'count'),
    ChurnRate=('Exited', 'mean')
).reset_index()
product_stats['RetentionScore'] = 1 - product_stats['ChurnRate']

# Weighted average retention score across all products, weighted by segment size
product_depth_index = (product_stats['RetentionScore'] * product_stats['Count']).sum() / product_stats['Count'].sum()

print(product_stats)
print(f"\nProduct Depth Index (weighted avg retention across product tiers): {product_depth_index:.4f}")

   NumOfProducts  Count  ChurnRate  RetentionScore
0              1   5084   0.277144        0.722856
1              2   4590   0.075817        0.924183
2              3    266   0.827068        0.172932
3              4     60   1.000000        0.000000

Product Depth Index (weighted avg retention across product tiers): 0.7963


In [4]:
high_balance_customers = df[df['HighBalance'] == 1]
high_balance_disengagement_rate = (high_balance_customers['IsActiveMember'] == 0).mean()

print(f"High-Balance Disengagement Rate: {high_balance_disengagement_rate:.4f} ({high_balance_disengagement_rate*100:.1f}%)")

High-Balance Disengagement Rate: 0.4912 (49.1%)


In [5]:
churn_no_card = df[df['HasCrCard']==0]['Exited'].mean()
churn_card = df[df['HasCrCard']==1]['Exited'].mean()

credit_card_stickiness_score = churn_no_card - churn_card
print(f"Churn (no card): {churn_no_card:.4f}")
print(f"Churn (has card): {churn_card:.4f}")
print(f"Credit Card Stickiness Score: {credit_card_stickiness_score:.4f}")

Churn (no card): 0.2081
Churn (has card): 0.2018
Credit Card Stickiness Score: 0.0063


In [7]:
# Normalize Tenure to 0-1
df['TenureNorm'] = df['Tenure'] / df['Tenure'].max()

# Product score: 2 products = 1.0 (best), 1 product = 0.5, 3-4 products = 0.0 (worst, per Phase 4 finding)
def product_score(n):
    if n == 2:
        return 1.0
    elif n == 1:
        return 0.5
    else:
        return 0.0

df['ProductScore'] = df['NumOfProducts'].apply(product_score)

df['RelationshipStrengthIndex'] = (
    0.4 * df['IsActiveMember'] +
    0.3 * df['ProductScore'] +
    0.1 * df['HasCrCard'] +
    0.2 * df['TenureNorm']
)

print(df['RelationshipStrengthIndex'].describe())

# Validate: does the index actually correlate with NOT churning?
correlation = df['RelationshipStrengthIndex'].corr(df['Exited'])
print(f"\nCorrelation with Exited: {correlation:.4f} (should be negative if index is well-designed)")



count    10000.000000
mean         0.590806
std          0.230259
min          0.000000
25%          0.400000
50%          0.590000
75%          0.800000
max          1.000000
Name: RelationshipStrengthIndex, dtype: float64

Correlation with Exited: -0.2700 (should be negative if index is well-designed)


In [8]:
kpi_by_geo = df.groupby('Geography').apply(lambda g: pd.Series({
    'ChurnRate': g['Exited'].mean(),
    'AvgRelationshipStrength': g['RelationshipStrengthIndex'].mean(),
    'HighBalanceDisengagement': (g[g['HighBalance']==1]['IsActiveMember']==0).mean()
}), include_groups=False)

print(kpi_by_geo.round(3))

           ChurnRate  AvgRelationshipStrength  HighBalanceDisengagement
Geography                                                              
France         0.162                    0.594                     0.500
Germany        0.324                    0.576                     0.504
Spain          0.167                    0.599                     0.445


## KPI Definitions Summary

- **Engagement Retention Ratio: 1.88** — inactive customers churn at nearly twice the rate of active customers (26.9% vs 14.3%).
- **Product Depth Index: 0.80** — masks a non-monotonic pattern underneath: retention peaks at 2 products (92.4%) and collapses at 3-4 products (17.3%, then 0%). The tier breakdown is more informative than the single index value.
- **High-Balance Disengagement Rate: 49.1%** — nearly half of all high-balance customers are inactive, independent of whether they've churned yet. A leading indicator of dormant premium relationships.
- **Credit Card Stickiness Score: 0.006** — effectively zero, confirming card ownership is not a retention lever.
- **Relationship Strength Index** (0.4×activity + 0.3×product fit + 0.1×card + 0.2×tenure): correlates -0.27 with churn, validating the weighting. Germany shows the lowest average score (0.576 vs 0.594–0.599 elsewhere), consistent with its higher churn rate throughout the analysis.